In [ ]:
import random
from collections import deque

import numpy as np
import yaml

from giotto.agents.algorithms.alphazero.train import AlphaZeroTrainer
from giotto.envs.tris import TrisEnv

This notebook replicate the training process step by step, to visualize it and check it.

# Initialize trainer

In [ ]:
env = TrisEnv()

with open("../agents/algorithms/alphazero/config_tris.yaml") as f:
    config = yaml.safe_load(f)

trainer = AlphaZeroTrainer(
    env=env,
    config=config,
    save_dir="logs",
    device="cpu",
)

iterations = int(trainer.config["training"]["iterations"])
buffer_size = int(trainer.config["training"]["buffer_size"])
games_per_iteration = int(trainer.config["training"]["games_per_iteration"])
batch_size = int(trainer.config["training"]["batch_size"])

training_steps_per_iteration = int(trainer.config["training"].get("training_steps_per_iteration", 0))

replay_buffer: deque[dict] = deque(maxlen=buffer_size)

# Training iteration

Self play

In [ ]:
new_records: list[dict] = []
for _ in range(10):
    new_records.extend(trainer.self_play_game())

new_records = trainer.augment_data(new_records)
replay_buffer.extend(new_records)

In [ ]:
print(len(replay_buffer))
print(replay_buffer[0])

In [ ]:
if training_steps_per_iteration <= 0:
    training_steps_per_iteration = max(1, len(replay_buffer) // batch_size)

losses_total = []
losses_policy = []
losses_value = []

for _ in range(training_steps_per_iteration):
    batch_records = random.sample(list(replay_buffer), batch_size)
    loss_dict = trainer.train_step(batch_records)
    losses_total.append(loss_dict["total_loss"])
    losses_policy.append(loss_dict["policy_loss"])
    losses_value.append(loss_dict["value_loss"])

avg_total = float(np.mean(losses_total))
avg_policy = float(np.mean(losses_policy))
avg_value = float(np.mean(losses_value))